In [1]:
import pandas as pd
import os
from sklearn.metrics import mean_squared_error, r2_score


BASE = "/Users/konuri/stacking/STACKING_TABLES"


files = {
    "RANDOM": "RANDOM_STACKING_TABLE.csv",
    "COLD_COMBINATION": "COLD_COMBINATION_STACKING_TABLE.csv",
    "COLD_CELL": "COLD_CELL_STACKING_TABLE.csv"
}


results = []


for split, file in files.items():

    print("\n====================")
    print(split)

    df = pd.read_csv(
        os.path.join(BASE,file)
    )


    y = df["y_true"]


    models = {

        "CatBoost": df["catboost_prediction"],

        "DMPNN": df["dmpnn_prediction"],

    }


    for name,pred in models.items():

        rmse = mean_squared_error(
            y,
            pred
        ) ** 0.5

        r2 = r2_score(
            y,
            pred
        )


        results.append(
            {
                "Split": split,
                "Model": name,
                "RMSE": rmse,
                "R2": r2
            }
        )


        print(
            name,
            "RMSE:",
            round(rmse,4),
            "R2:",
            round(r2,4)
        )


results_df = pd.DataFrame(results)


print("\nFINAL BASE MODEL COMPARISON")
display(results_df)


results_df.to_csv(
    f"{BASE}/BASE_MODEL_METRICS.csv",
    index=False
)

print("Saved")


RANDOM
CatBoost RMSE: 7.3815 R2: -0.109
DMPNN RMSE: 5.9026 R2: 0.2909

COLD_COMBINATION
CatBoost RMSE: 7.42 R2: -0.1112
DMPNN RMSE: 6.6127 R2: 0.1175

COLD_CELL
CatBoost RMSE: 8.4149 R2: -0.0403
DMPNN RMSE: 7.13 R2: 0.2531

FINAL BASE MODEL COMPARISON


,Split,Model,RMSE,R2
0,RANDOM,CatBoost,7.381524,-0.108951
1,RANDOM,DMPNN,5.902640,0.290891
2,COLD_COMBINATION,CatBoost,7.420048,-0.111192
3,COLD_COMBINATION,DMPNN,6.612668,0.117471
4,COLD_CELL,CatBoost,8.414944,-0.040310
5,COLD_CELL,DMPNN,7.130012,0.253137


Saved


In [2]:
import pandas as pd
import os


BASE = "/Users/konuri/stacking/STACKING_TABLES"


# Load metrics
base = pd.read_csv(
    f"{BASE}/BASE_MODEL_METRICS.csv"
)

meta = pd.read_csv(
    f"{BASE}/META_MODEL_RESULTS.csv"
)


print("BASE MODEL RESULTS")
display(base)

print("META MODEL RESULTS")
display(meta)


# Keep best meta model per split
meta_best = (
    meta
    .sort_values(
        "R2",
        ascending=False
    )
    .groupby("split")
    .first()
    .reset_index()
)


# Rename for merge
base_pivot = (
    base
    .pivot(
        index="Split",
        columns="Model",
        values=["RMSE","R2"]
    )
)


base_pivot.columns = [
    "_".join(col)
    for col in base_pivot.columns
]


base_pivot = base_pivot.reset_index()


final = base_pivot.merge(
    meta_best,
    left_on="Split",
    right_on="split",
    how="left"
)


final = final.drop(
    columns=["split"],
    errors="ignore"
)


print("\n==============================")
print("FINAL TRUSTSYN ENSEMBLE TABLE")
display(final)


final.to_csv(
    f"{BASE}/FINAL_TRUSTSYN_ENSEMBLE_RESULTS.csv",
    index=False
)


print(
    "\nSaved:",
    f"{BASE}/FINAL_TRUSTSYN_ENSEMBLE_RESULTS.csv"
)

BASE MODEL RESULTS


,Split,Model,RMSE,R2
0,RANDOM,CatBoost,7.381524,-0.108951
1,RANDOM,DMPNN,5.902640,0.290891
2,COLD_COMBINATION,CatBoost,7.420048,-0.111192
3,COLD_COMBINATION,DMPNN,6.612668,0.117471
4,COLD_CELL,CatBoost,8.414944,-0.040310
5,COLD_CELL,DMPNN,7.130012,0.253137


META MODEL RESULTS


,split,model,RMSE,R2
0,RANDOM,Ridge,5.888001,0.294404
1,RANDOM,XGBoost,5.910889,0.288907
2,COLD_COMBINATION,Ridge,6.578213,0.126644
3,COLD_COMBINATION,XGBoost,6.568226,0.129294
4,COLD_CELL,Ridge,7.054416,0.268890
5,COLD_CELL,XGBoost,6.788992,0.322872



FINAL TRUSTSYN ENSEMBLE TABLE


,Split,RMSE_CatBoost,RMSE_DMPNN,R2_CatBoost,R2_DMPNN,model,RMSE,R2
0,COLD_CELL,8.414944,7.130012,-0.040310,0.253137,XGBoost,6.788992,0.322872
1,COLD_COMBINATION,7.420048,6.612668,-0.111192,0.117471,XGBoost,6.568226,0.129294
2,RANDOM,7.381524,5.902640,-0.108951,0.290891,Ridge,5.888001,0.294404



Saved: /Users/konuri/stacking/STACKING_TABLES/FINAL_TRUSTSYN_ENSEMBLE_RESULTS.csv
